In [15]:
from langchain_core.document_loaders.base import BaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage,SystemMessage
import os
import yt_dlp
from pathlib import Path
import webvtt

### Custom Document Loader For fetching Transcript From YouTube Videos


In [ ]:
# Creating a custom document loader that fetches the transcript from the youtube video using yt-dlp and parse it using webvtt

class TranscriptLoader(BaseLoader):

    def __init__(self, video_url:str, language:str="en"):
        self.video_url=video_url
        self.language=language
        self.temp_filename="temp_transcript"
    

    def load(self)->list[Document]:

        # configure the ydl library to get the desired output
        ydl_opts = {
        
        # 'cookiefile':'your-youtube-cookies.txt file',

        # only uncomment this if your model is not running or asking for captcha verification, to do this you must download the cookies.txt file by any cookies.txt locally downloader.

        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': [self.language],
        'subtitlesformat': 'vtt',
        'outtmpl': self.temp_filename,
        'quiet': True,
        'no_warnings': True
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:

            ydl.download([self.video_url])

        expected_file=f"{self.temp_filename}.{self.language}.vtt"
        

        clean_line=[]
        if Path(expected_file).exists():
            for captions in webvtt.read(expected_file):
                lines=captions.text.strip().split('\n')

                # Sometimes youtube combines lines together with a \n so the first step is to ensure that every text senetence is being split into each individual sentence.

                for line in lines:
                    line=line.strip()
                    if not line:
                        continue

                    if not clean_line:
                        clean_line.append(line)

                        #  If the list is empty than directly append the line into the list

                    else:
                        last_line=clean_line[-1]

                        if line==last_line:
                            continue

                        # Checking if the new line and the previous line from the library are same than continue and do nothing. 

                        if line.startswith(last_line):
                            clean_line[-1]=line

                        # check if the new line starts with last line than replace the last line with new line
                        elif last_line not in line:
                            clean_line.append(line)

                        # Else append new line directly.

            os.remove(expected_file)
        else:
            raise FileNotFoundError(f"Transcript is not available for the provided video {self.video_url}")
        
        metadata={"video_url":self.video_url, "language":self.language}

        transcript_text=" ".join(clean_line)

        return [Document(
            page_content=transcript_text.strip(),
            metadata=metadata
        )]
        





In [4]:
transcript=TranscriptLoader(video_url="https://youtu.be/-gy78kmy0Jg?si=ps8nWEnxnGenl6dl", language="en")
result=transcript.load()

In [5]:
print(result[0].page_content)

There are already parasites in society who attack you people who attack the system. People look at this thing. This is a cockroach. And right now in May 2026, 22 million Indians have decided that this is their political symbol. The youth of this country is fed up of all existing political parties and now they want their own political platform where they can raise their own voice. In just 6 days, a parody party called the cockroach party got more Instagram followers than the BJP itself. More than the world's largest political party. Now, most of the internet is asking the wrong question. They're asking, is CJP a real party? Will it contest elections? Is this Jenz's revolt? But do you realize that is not the story? The real story is this. How does a country of 1.4 billion people get so angry that a cockroach becomes more popular than the ruling party itself and that too in less than a week? I heard about it few days ago and the initiative is really good. They are asking REAL QUESTIONS. T

## Splitting The Text Into Sentences

In [6]:

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([result[0].page_content])

In [23]:
chunks

[Document(metadata={}, page_content="There are already parasites in society who attack you people who attack the system. People look at this thing. This is a cockroach. And right now in May 2026, 22 million Indians have decided that this is their political symbol. The youth of this country is fed up of all existing political parties and now they want their own political platform where they can raise their own voice. In just 6 days, a parody party called the cockroach party got more Instagram followers than the BJP itself. More than the world's largest political party. Now, most of the internet is asking the wrong question. They're asking, is CJP a real party? Will it contest elections? Is this Jenz's revolt? But do you realize that is not the story? The real story is this. How does a country of 1.4 billion people get so angry that a cockroach becomes more popular than the ruling party itself and that too in less than a week? I heard about it few days ago and the initiative is really go

## Initializing the chroma vector store and creating the retriever

In [7]:
model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',model_kwargs={'device':0})

vectorstore=Chroma.from_documents(
    embedding=model,
    documents=chunks

)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4044.61it/s]


### Retrieving the relevant chunks from the vector store and passing it to the LLM for generating the answer

In [27]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":10}
)

In [9]:
def generatecontext(data):
    context="\n\n".join(docs.page_content for docs in data)
    return context


In [32]:
message=[]

In [33]:
query="Give me a brief summary "
message.append(HumanMessage(query))

raw_context=retriever.invoke(query)


In [34]:
context=generatecontext(raw_context)
message.append(SystemMessage(context))

### Designing the prompt template for the LLM to generate the answer based on the retrieved chunks from the vector store

In [35]:
model=GoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [36]:
result=model.invoke(message)
result

'This text argues that India’s **"First-Past-The-Post" (FPTP)** voting system is the root cause of many of the country\'s political issues. Here is a brief summary of the main points:\n\n**The Core Argument:**\nThe author contends that the FPTP system—a colonial legacy—is a "virus" that undermines Indian democracy by failing to represent the majority of voters and incentivizing divisive politics.\n\n**The Three "Viruses" of FPTP:**\n1.  **Disenfranchisement:** Because candidates only need a plurality (not a majority) to win, the preferences of the majority (often 60%+) are effectively ignored.\n2.  **Caste-Based Politics:** To win, candidates focus on "packing" specific demographic groups into one constituency rather than appealing to the general public. This encourages identity politics over policy.\n3.  **The Death of National Issues:** Issues that affect all Indians (like climate change or MSME support) are ignored because they aren\'t geographically concentrated. A party can have m